# 最適化と正則化

最適化と正則化は、損失が下がる理由を『式』と『数値変化』の両方で確認する構成です。


## 背景と目的

高性能モデルでも最適化が不安定だと収束せず、汎化しない解に落ちることがあります。

最適化手法と正則化を使い分けられると、学習安定性と汎化性能を両立できます。

ハイパーパラメータの変更が挙動に与える影響を比較します。


## 最初に解きたい疑問

1. 同じ初期点から `plain / weight_decay / small_lr` を比べると、何が見えるのか。
2. `data_loss` と `total_loss` は、どう見分ければよいのか。
3. 勾配降下法とミニバッチ更新は、何が同じで何が違うのか。
4. 正則化はなぜ『精度を上げる方法』ではなく『崩れを抑える方法』として扱われるのか。
5. weight decay と学習率を小さくすることは、何が違うのか。


## 先に押さえる言葉

- `収束`: 更新を続けても値が大きく暴れず、ある状態に落ち着いていくことです。学習が進むかどうかの基本的な目安になります。
- `正則化`: モデルが複雑になりすぎるのを抑えるための仕組みです。訓練データに合わせすぎるのを防ぎます。
- `weight decay`: 重みを少しずつ小さくする正則化の考え方です。L2 正則化と近い働きをします。
- `汎化`: 学習に使っていないデータでも、うまく予測できる性質です。訓練損失が低いだけでは足りません。


## 実行前の見取り図

1. 同じ初期点から始めた `plain`, `weight_decay`, `small_lr` で、`data_loss` と `weight_norm` がどう変わるかを見る。
2. `weight decay` が更新式に入ると、単に loss に penalty を後から足すだけの場合と何が違うかを見る。
3. ミニバッチで平均損失と平均勾配を作ると、更新がどう変わるかを見る。


## 出力の読み方

- `norm_plain/decay` の差が小さい/大きいと何を意味するか。


## つまずきやすい点

- `theta <- theta - eta ∇L` が、ただの式変形ではなく最適化そのものになっている理由の説明が足りない。
- 正則化の良し悪しを訓練データだけで判断してはいけない理由と、検証データを使う意味の説明が足りない。
- `data_loss` と `total_loss` の役割分担の注意。
- 正則化の本質が、単発の loss ではなく長い学習軌道で効くこと。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- このノートの比較は1ステップ toy 更新で、汎化性能の比較ではないこと。


In [ ]:
import math


## 実験 1: 最適化の初期点を定義する

最適化手法比較のため、重みと予測を同じ初期点から開始します。


In [ ]:
x = [0.9, -0.3, 0.1]
w = [0.5, -0.2, 0.4]
b = 0.03
z = sum(xi * wi for xi, wi in zip(x, w)) + b
y = 1 / (1 + math.exp(-z))
print('task = optimization-regularization', 'init_pred=', round(y, 6))


同じ初期点を使うことで、手法差を比較しやすくします。



## 実験 2: 損失を計算する

次に、予測値に対する損失を計算します。学習は損失を下げる方向に進むので、損失の意味を言葉で説明できることが重要です。


In [ ]:
target = 1.0
eps = 1e-9
loss = -(target * math.log(y + eps) + (1 - target) * math.log(1 - y + eps))
print('loss =', round(loss, 6))
print('error =', round(target - y, 6))

損失は『どれだけ外したか』を測る物差しです。物差しがない状態では、改善の方向を決められません。



## 式と実装の往復

1. $z = W x + b$
2. $\theta \leftarrow \theta - \eta \, \nabla_{\theta} L$

3. BCE + sigmoid のとき、出力前活性 `z` に対する勾配は $y - target$


## 実験 3: 勾配降下法の一歩

ここで、重みを一回更新する最小実験を行います。更新前後の損失を比較して、学習の方向が正しいかを確認します。


In [ ]:
lr = 0.1
grad_z = y - target  # BCE + sigmoid のとき dL/dz
w_new = [wi - lr * grad_z * xi for wi, xi in zip(w, x)]
b_new = b - lr * grad_z
z_new = sum(xi * wi for xi, wi in zip(x, w_new)) + b_new
y_new = 1 / (1 + math.exp(-z_new))
print('grad_z =', round(grad_z, 6))
print('b before/after =', round(b, 6), round(b_new, 6))
print('y before/after =', round(y, 6), round(y_new, 6))


この更新で損失が下がれば、勾配方向が合理的だったと言えます。下がらないなら学習率や符号を疑います。



ここでは重みだけでなく bias も同時に更新し、BCE と勾配の対応が崩れないようにしています。


## 実験 4: weight decay を更新式に入れる

ここでは通常更新、weight decay あり、学習率だけ小さくした更新を同じ初期点から比べます。正則化が loss の表示だけでなく、実際の更新方向をどう変えるかを確かめます。


In [ ]:
grad_w = [grad_z * xi for xi in x]


def predict_prob(sample, weights, bias):
    z_cur = sum(si * wi for si, wi in zip(sample, weights)) + bias
    return 1 / (1 + math.exp(-z_cur))


def bce_value(prob, target):
    return -(target * math.log(prob + eps) + (1 - target) * math.log(1 - prob + eps))


l2 = 0.1
small_lr = 0.03

before_prob = predict_prob(x, w, b)
before_loss = bce_value(before_prob, target)
before_norm = math.sqrt(sum(wi * wi for wi in w))
print('before:', 'pred=', round(before_prob, 6), 'data_loss=', round(before_loss, 6), 'weight_norm=', round(before_norm, 6))

updates = [
    ('plain', [wi - lr * gwi for wi, gwi in zip(w, grad_w)], b - lr * grad_z, 0.0),
    ('weight_decay', [wi - lr * (gwi + l2 * wi) for wi, gwi in zip(w, grad_w)], b - lr * grad_z, l2),
    ('small_lr', [wi - small_lr * gwi for wi, gwi in zip(w, grad_w)], b - small_lr * grad_z, 0.0),
]

for name, ww, bb, penalty in updates:
    prob = predict_prob(x, ww, bb)
    data_loss = bce_value(prob, target)
    weight_norm = math.sqrt(sum(wi * wi for wi in ww))
    total_loss = data_loss + 0.5 * penalty * sum(wi * wi for wi in ww)
    print(name, 'pred=', round(prob, 6), 'data_loss=', round(data_loss, 6), 'weight_norm=', round(weight_norm, 6), 'total_loss=', round(total_loss, 6))

weight decay は `grad + l2 * w` として更新式に入り、重みが大きい方向を追加で押し戻します。学習率を小さくするだけでは全方向の step を一様に縮めるだけなので、同じ動きにはなりません。



## 実験 5: ミニバッチで平均損失と平均勾配を見る

最後に、複数サンプルをまとめて扱う発想を確認します。ここでは各サンプルの損失と勾配を平均してから 1 回だけ更新します。


In [ ]:
batch = [[0.8, -0.4, 0.2], [0.2, 0.1, -0.3], [0.5, -0.2, 0.7]]
targets = [1.0, 0.0, 1.0]
sample_preds = []
sample_losses = []
sample_grad_z = []

for bx, bt in zip(batch, targets):
    z_b = sum(xi * wi for xi, wi in zip(bx, w_new)) + b_new
    y_b = 1 / (1 + math.exp(-z_b))
    loss_b = bce_value(y_b, bt)
    sample_preds.append(y_b)
    sample_losses.append(loss_b)
    sample_grad_z.append(y_b - bt)

grad_w_batch = [
    sum(g * bx[j] for g, bx in zip(sample_grad_z, batch)) / len(batch)
    for j in range(len(w_new))
]
grad_b_batch = sum(sample_grad_z) / len(batch)

plain_batch_w = [wi - lr * gw for wi, gw in zip(w_new, grad_w_batch)]
plain_batch_b = b_new - lr * grad_b_batch
decay_batch_w = [wi - lr * (gw + l2 * wi) for wi, gw in zip(w_new, grad_w_batch)]
decay_batch_b = b_new - lr * grad_b_batch


def mean_batch_loss(weights, bias):
    vals = []
    for bx, bt in zip(batch, targets):
        z_b = sum(xi * wi for xi, wi in zip(bx, weights)) + bias
        y_b = 1 / (1 + math.exp(-z_b))
        vals.append(bce_value(y_b, bt))
    return sum(vals) / len(vals)


print('sample_preds =', [round(p, 4) for p in sample_preds])
print('sample_losses =', [round(v, 4) for v in sample_losses])
print('batch_loss_mean_before =', round(sum(sample_losses) / len(sample_losses), 6))
print('grad_b_batch =', round(grad_b_batch, 6))
print('grad_w_batch =', [round(v, 6) for v in grad_w_batch])
print('batch_loss_mean_plain_after =', round(mean_batch_loss(plain_batch_w, plain_batch_b), 6))
print('batch_loss_mean_decay_after =', round(mean_batch_loss(decay_batch_w, decay_batch_b), 6))
print('norm_plain/decay =', round(math.sqrt(sum(wi * wi for wi in plain_batch_w)), 6), round(math.sqrt(sum(wi * wi for wi in decay_batch_w)), 6))


ミニバッチでは各サンプルの損失と勾配を平均してから 1 回更新します。ここに weight decay を重ねると、平均勾配に加えて重みの大きさも同時に押し戻せるので、更新後のノルムの違いとして観察できます。



## 振り返り

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 学習率が大きすぎて発散する
2. 検証損失の監視をせず過学習を見逃す
3. 前処理と活性化の相性を無視する
